**Objetivo**

Promover os dados de alunos da camada Bronze para a Silver, padronizando chaves e categorias e preservando a rastreabilidade da ingestão.

**Fonte de dados**

- `bronze.aluno`

**Destino**

- `silver.aluno`

**Decisões confirmadas com a amostra**

- `id_municipio` possui 7 dígitos;
- `id_escola` é um identificador mascarado de 8 dígitos, não o código INEP real;
- `presenca`, `preenchimento_caderno` e `alfabetizado` são indicadores binários 0/1;
- os indicadores preservam o inteiro em colunas `*_id` e são materializados como `BOOLEAN`;
- a rede preserva o código em `rede_id` e adiciona sua descrição;
- nulos em `proficiencia` e `peso_aluno` são avaliados de forma condicional ao preenchimento da prova.

> A amostra de 200 linhas é concentrada em `caderno=1`, `serie=2` e alunos sem caderno preenchido. Por isso, os domínios continuam sendo diagnosticados sobre a tabela completa antes da escrita.

## 0. Configurando sessão spark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_aluno")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# Projeto usado para faturamento das consultas
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

:: loading settings :: url = jar:file:/opt/micromamba/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jupyter/.ivy2.5.2/cache
The jars for the packages stored in: /home/jupyter/.ivy2.5.2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0f4174b2-bffc-48f6-84c4-16ec8071cdd7;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 in central
:: resolution report :: resolve 537ms :: artifacts dl 7ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------------

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [4]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [17]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_bronze_aluno = f"{par_source_project}.bronze.aluno"
par_source_silver_aluno = f"{par_source_project}.silver.aluno"

colunas_esperadas = [
    "ano", "id_municipio", "id_escola", "id_aluno", "caderno",
    "serie", "rede", "presenca", "preenchimento_caderno",
    "alfabetizado", "proficiencia", "peso_aluno",
    "_ingestao_timestamp", "_fonte"
]

colunas_categoricas = [
    "caderno", "serie", "rede", "presenca",
    "preenchimento_caderno", "alfabetizado"
]

cadernos_validos = [str(valor) for valor in range(1, 22)] + ["43"]
series_validas = [2]
redes_validas = ["2", "3", "4"]

rede_map = {
    "0": "Total (Federal, Estadual, Municipal e Privada)",
    "1": "Federal",
    "2": "Estadual",
    "3": "Municipal",
    "4": "Privada",
    "5": "Pública (Estadual e Municipal)",
    "6": "Pública (Federal, Estadual e Municipal)",
}

## 3. Leitura dos dados da origem

In [8]:
df_src_alunos = (
    spark.read.format("bigquery")
    .option("table", par_source_bronze_aluno)
    .load()
)

### 3.1 Validação do contrato de entrada

In [11]:
colunas_ausentes = sorted(set(colunas_esperadas) - set(df_src_alunos.columns))
if colunas_ausentes:
    raise ValueError(f"Schema inválido. Colunas ausentes na Bronze: {colunas_ausentes}")

df_alunos = df_src_alunos.select(*colunas_esperadas)

# A Bronze pode representar os indicadores como STRING ou INTEGER.
# O contrato valida o domínio, não o tipo físico recebido do conector.
colunas_indicadoras = ["presenca", "preenchimento_caderno", "alfabetizado"]
expressoes_invalidas = []
for coluna in colunas_indicadoras:
    valor = F.trim(F.col(coluna).cast("string"))
    expressoes_invalidas.append(
        F.sum(
            F.when(F.col(coluna).isNotNull() & ~valor.isin("0", "1"), 1).otherwise(0)
        ).alias(coluna)
    )

invalidos_por_indicador = df_alunos.agg(*expressoes_invalidas).first().asDict()
invalidos_por_indicador = {
    coluna: quantidade
    for coluna, quantidade in invalidos_por_indicador.items()
    if quantidade > 0
}
if invalidos_por_indicador:
    raise ValueError(
        f"Indicadores fora do domínio booleano {{0, 1}}: {invalidos_por_indicador}"
    )

In [13]:
# Verificação dos dominios das colunas categoricas - Check das booleanas e outras com mais categorias
for coluna in colunas_categoricas:
    print(f"=== Domínio de {coluna} ===")
    (
        df_alunos
        .groupBy(F.col(coluna))
        .count()
        .orderBy(F.desc("count"), F.asc_nulls_first(coluna))
        .show(100, truncate=False)
    )

=== Domínio de caderno ===


+-------+------+
|caderno|count |
+-------+------+
|1      |250518|
|2      |188389|
|4      |188161|
|3      |187940|
|7      |187939|
|6      |187890|
|5      |187762|
|8      |180992|
|15     |180584|
|14     |178493|
|10     |178458|
|13     |178241|
|11     |178141|
|16     |177848|
|12     |177779|
|9      |177689|
|18     |177089|
|20     |176372|
|17     |176203|
|21     |176040|
|19     |175453|
|43     |18    |
+-------+------+

=== Domínio de serie ===


+-----+-------+
|serie|count  |
+-----+-------+
|2    |3867999|
+-----+-------+

=== Domínio de rede ===


+----+-------+
|rede|count  |
+----+-------+
|3   |3432576|
|2   |435398 |
|4   |25     |
+----+-------+

=== Domínio de presenca ===
+--------+-------+
|presenca|count  |
+--------+-------+
|1       |3355846|
|0       |512153 |
+--------+-------+

=== Domínio de preenchimento_caderno ===
+---------------------+-------+
|preenchimento_caderno|count  |
+---------------------+-------+
|1                    |3354661|
|0                    |513338 |
+---------------------+-------+

=== Domínio de alfabetizado ===
+------------+-------+
|alfabetizado|count  |
+------------+-------+
|1           |1984546|
|0           |1883453|
+------------+-------+



## 4. Transformações

### 4.1 Padronização de chaves, tipos e categorias

In [14]:
def texto_normalizado(coluna):
    valor = F.trim(F.col(coluna).cast("string"))
    return F.when(valor == "", F.lit(None)).otherwise(valor)

id_municipio_limpo = texto_normalizado("id_municipio")
map_rede = F.create_map([F.lit(x) for item in rede_map.items() for x in item])

def inteiro_para_booleano(coluna):
    return (
        F.when(F.col(coluna) == 1, F.lit(True))
        .when(F.col(coluna) == 0, F.lit(False))
        .otherwise(F.lit(None).cast("boolean"))
    )

df_silver_alunos = (
    df_alunos
    .withColumn("ano", F.col("ano").cast("int"))
    # Código IBGE: completa com zero apenas quando o conteúdo já é numérico.
    .withColumn(
        "id_municipio",
        F.when(
            id_municipio_limpo.rlike("^[0-9]{1,7}$"),
            F.lpad(id_municipio_limpo, 7, "0")
        ).otherwise(id_municipio_limpo)
    )
    # Escola e aluno são identificadores opacos: não converter para número nem preencher zeros.
    .withColumn("id_escola", texto_normalizado("id_escola"))
    .withColumn("id_aluno", texto_normalizado("id_aluno"))
    .withColumn("caderno", F.upper(texto_normalizado("caderno")))
    .withColumn("serie", texto_normalizado("serie").cast("int"))
    .withColumnRenamed("rede", "rede_id")
    .withColumn("rede_id", F.upper(texto_normalizado("rede_id")))
    .withColumn("rede", map_rede[F.col("rede_id")])
    .withColumnRenamed("presenca", "presenca_id")
    .withColumn("presenca_id", F.col("presenca_id").cast("int"))
    .withColumn("presenca", inteiro_para_booleano("presenca_id"))
    .withColumnRenamed("preenchimento_caderno", "preenchimento_caderno_id")
    .withColumn("preenchimento_caderno_id", F.col("preenchimento_caderno_id").cast("int"))
    .withColumn("preenchimento_caderno", inteiro_para_booleano("preenchimento_caderno_id"))
    .withColumnRenamed("alfabetizado", "alfabetizado_id")
    .withColumn("alfabetizado_id", F.col("alfabetizado_id").cast("int"))
    .withColumn("alfabetizado", inteiro_para_booleano("alfabetizado_id"))
    .withColumn("proficiencia", F.col("proficiencia").cast("double"))
    .withColumn("peso_aluno", F.col("peso_aluno").cast("double"))
    .withColumn("_ingestao_timestamp", F.col("_ingestao_timestamp").cast("timestamp"))
    .withColumn("_fonte", texto_normalizado("_fonte"))
)
    

### 4.2 Remoção de valores duplicados

In [15]:
# Chave natural observada: um aluno por ano de aplicação da avaliação.
# Não há regra de versionamento por _ingestao_timestamp nesta promoção.
chave = ["ano", "id_aluno"]
df_silver_alunos_antes_dedup = df_silver_alunos
df_silver_alunos = df_silver_alunos.dropDuplicates(chave)

df_silver_alunos = df_silver_alunos.withColumn(
    "_silver_timestamp", F.current_timestamp()
)

## 5. Validação da qualidade

In [18]:
print("=== Relatório de Qualidade — silver.aluno ===")

qtd_bronze = df_src_alunos.count()
qtd_silver = df_silver_alunos.count()

# 1) Duplicidade na chave natural antes e depois do dropDuplicates
dups_antes = (
    df_silver_alunos_antes_dedup.groupBy(*chave).count()
    .filter(F.col("count") > 1).count()
)
dups_silver = (
    df_silver_alunos.groupBy(*chave).count().filter(F.col("count") > 1).count()
)
print(f"Chaves duplicadas antes da deduplicação: {dups_antes}")
print(f"Chaves duplicadas após deduplicação: {dups_silver}")

# 2) Formato das chaves
id_municipio_invalido = df_silver_alunos.filter(
    F.col("id_municipio").isNotNull()
    & ~F.col("id_municipio").rlike("^[0-9]{7}$")
).count()
print(f"id_municipio fora do padrão de 7 dígitos: {id_municipio_invalido}")

id_escola_invalido = df_silver_alunos.filter(
    F.col("id_escola").isNotNull()
    & ~F.col("id_escola").rlike("^[0-9]{8}$")
).count()
print(f"id_escola mascarado fora do formato observado de 8 dígitos: {id_escola_invalido}")

# 3) Nulos nas colunas da chave e de auditoria
colunas_criticas = list(dict.fromkeys(
    chave + [
        "serie", "rede_id", "presenca_id", "preenchimento_caderno_id",
        "alfabetizado_id", "_ingestao_timestamp", "_fonte"
    ]
))
for coluna in colunas_criticas:
    quantidade = df_silver_alunos.filter(F.col(coluna).isNull()).count()
    print(f"Nulos em '{coluna}': {quantidade}")

# 4) Domínios categóricos observados na tabela completa
dominios_invalidos = {
    "caderno": df_silver_alunos_antes_dedup.filter(
        F.col("caderno").isNotNull() & ~F.col("caderno").isin(*cadernos_validos)
    ).count(),
    "serie": df_silver_alunos_antes_dedup.filter(
        F.col("serie").isNotNull() & ~F.col("serie").isin(*series_validas)
    ).count(),
    "rede_id": df_silver_alunos_antes_dedup.filter(
        F.col("rede_id").isNotNull() & ~F.col("rede_id").isin(*redes_validas)
    ).count(),
}
for coluna, quantidade in dominios_invalidos.items():
    print(f"Valores fora do domínio esperado em {coluna}: {quantidade}")

if any(quantidade > 0 for quantidade in dominios_invalidos.values()):
    raise ValueError(
        f"Promoção interrompida: domínios categóricos inesperados: {dominios_invalidos}"
    )

qtd_caderno_43 = df_silver_alunos_antes_dedup.filter(F.col("caderno") == "43").count()
print(f"Registros com o código raro caderno=43: {qtd_caderno_43}")

# 5) Códigos de rede ainda não mapeados
print("Códigos de rede não mapeados:")
(
    df_silver_alunos
    .filter(F.col("rede_id").isNotNull() & F.col("rede").isNull())
    .groupBy("rede_id").count().orderBy(F.desc("count"))
    .show(100, truncate=False)
)

# Indicadores fora do domínio booleano aceito {0, 1}.
indicadores_fora_dominio_total = 0
for coluna_id, coluna_descricao in [
    ("presenca_id", "presenca"),
    ("preenchimento_caderno_id", "preenchimento_caderno"),
    ("alfabetizado_id", "alfabetizado"),
]:
    nao_mapeados = (
        df_silver_alunos
        .filter(F.col(coluna_id).isNotNull() & F.col(coluna_descricao).isNull())
        .count()
    )
    indicadores_fora_dominio_total += nao_mapeados
    print(f"Valores fora de {{0, 1}} em {coluna_id}: {nao_mapeados}")

if indicadores_fora_dominio_total > 0:
    raise ValueError(
        "Promoção interrompida: indicadores booleanos contêm valores fora de {0, 1}."
    )

# 6) Medidas inválidas. São reportadas, não descartadas automaticamente.
proficiencia_invalida = df_silver_alunos.filter(
    F.isnan("proficiencia") | (F.col("proficiencia") < 0)
).count()
peso_invalido = df_silver_alunos.filter(
    F.isnan("peso_aluno") | (F.col("peso_aluno") <= 0)
).count()
print(f"proficiencia negativa ou NaN: {proficiencia_invalida}")
print(f"peso_aluno não positivo ou NaN: {peso_invalido}")

metricas_ausentes_com_caderno_preenchido = df_silver_alunos.filter(
    (F.col("preenchimento_caderno_id") == 1)
    & (F.col("proficiencia").isNull() | F.col("peso_aluno").isNull())
).count()
print(
    "Linhas com caderno preenchido e proficiencia/peso ausente: "
    f"{metricas_ausentes_com_caderno_preenchido}"
)

inconsistencia_presenca = df_silver_alunos.filter(
    (F.col("presenca_id") == 0)
    & (F.col("preenchimento_caderno_id") == 1)
).count()
inconsistencia_alfabetizado = df_silver_alunos.filter(
    (F.col("alfabetizado_id") == 1)
    & (F.col("preenchimento_caderno_id") != 1)
).count()
print(f"Ausente com caderno preenchido: {inconsistencia_presenca}")
print(f"Alfabetizado sem caderno preenchido: {inconsistencia_alfabetizado}")

# 7) Consistência escola/município no mesmo ano
escolas_multiplos_municipios = (
    df_silver_alunos
    .filter(F.col("id_escola").isNotNull())
    .groupBy("ano", "id_escola")
    .agg(F.countDistinct("id_municipio").alias("qtd_municipios"))
    .filter(F.col("qtd_municipios") > 1)
    .count()
)
print(f"Escolas associadas a mais de um município no mesmo ano: {escolas_multiplos_municipios}")
print(f"Linhas Bronze: {qtd_bronze} -> Linhas Silver: {qtd_silver}")

=== Relatório de Qualidade — silver.aluno ===


Chaves duplicadas antes da deduplicação: 0
Chaves duplicadas após deduplicação: 0


id_municipio fora do padrão de 7 dígitos: 0


id_escola mascarado fora do formato observado de 8 dígitos: 0
Nulos em 'ano': 0
Nulos em 'id_aluno': 0


Nulos em 'serie': 0


...

proficiencia negativa ou NaN: 0
peso_aluno não positivo ou NaN: 0


Linhas com caderno preenchido e proficiencia/peso ausente: 0


Ausente com caderno preenchido: 0
Alfabetizado sem caderno preenchido: 0


[Stage 266:==============>                                          (1 + 3) / 4]

Escolas associadas a mais de um município no mesmo ano: 0
Linhas Bronze: 3867999 -> Linhas Silver: 3867999


## 6. Armazenamento no BQ

In [19]:
(
    df_silver_alunos.write.format("bigquery")
    .option("table", par_source_silver_aluno)
    .option("writeMethod", "direct")
    .option("clusteredFields", "ano,id_municipio,id_escola,rede_id")
    .mode("overwrite")
    .save()
)

26/08/24 01:24:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                